In [2]:
from weights import WeightMatrix
from topologies import square_torus
import numpy as np
import warnings
warnings.filterwarnings("error")


In [ ]:
square_torus(25)

In [3]:
def step_simulation(R, V, I, t=0, Delta=1, Eta=-5, J=15, tau=1, dt=1e-3):
    # Heun (RK2) instead of Euler + finite guards
    fR = Delta/np.pi + 2*R*V
    fV = V**2 + Eta + J*R + I - (np.pi**2)*(R**2)  # NOTE: +J*R (match your LaTeX)
    R1 = R + dt*fR
    V1 = V + np.clip(dt*fV, -1e8, 1e8)
    fR1 = Delta/np.pi + 2*R1*V1
    fV1 = V1**2 + Eta + J*R1 + I - (np.pi**2)*(R1**2)
    dR = 0.5*dt*(fR + fR1)
    np.nan_to_num(dR, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    #dR = np.clip(dR, -R/10, 1)
    R += dR*(10-R)/10
    R[:] = np.clip(R, 1e-5, 10)
    dV = 0.5*dt*(fV + fV1)*(100-np.abs(V))/100
    np.nan_to_num(dV, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    #dV = np.clip(dV, -1, 1)
    V += dV
    #V[:] = np.clip(V, -1e2, 1e2)

    np.nan_to_num(R, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    np.nan_to_num(V, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    return t + dt, dR


In [4]:
import threading, queue
import ipywidgets as w

Z = 10

t = 0
weights = WeightMatrix(square_torus(Z),
                       weight_initializer=lambda **s: (np.random.normal(size=s['size'], loc=1, scale=0.012)))
children = np.array([*weights.network.values()])

R, V = np.zeros((2, weights.size))
V.fill(-100)
R.fill(0)
I = np.zeros(weights.size)
weights.check = True




def update_I(I, V, weights, t, tau=2, dt=1e-2):
    I[0:Z] = 3 * np.sin(np.pi * t / 20)
    I[-Z:0] = 5 * np.sin(np.pi * t / 20)

    children = np.array([*weights.network.values()])
    delta = (R[:,None] * weights[np.arange(weights.size)[:, None], children] - I[:,None]/2)/tau
    delta *= dt
    np.add.at(I, children.ravel(), delta.ravel())
    I[0:Z] = 10 * np.sin(np.pi * t)
    I[-Z:] = 10 * np.sin(np.pi * (t-1))
    I[Z**2//2:Z**2//2+Z]= 10 * np.sin(np.pi * (t-1/2))
    I += np.random.normal(size=weights.size, loc=.01, scale=0.05)
    


def update_W(weights, R, dR, A_p=5e-3, A_m=1e-3, tau_p=0.5, tau_m=.25):
    i = np.arange(weights.size)[:, None]
    j = np.array([*weights.network.values()])
    weights.at[i, j] <<= A_p * R[:, None]*(R[j] + tau_p * dR[j]) - A_m * R[j]*(R[:,None] - tau_m*dR[:, None]) - 1e-2*np.clip(weights[i,j],-1e7,1e7)**3


import plotly.graph_objects as go, time

# initial setup
fig = go.FigureWidget()
fig.update_layout(width=500, height=500, margin=dict(l=0, r=0, b=0, t=0))

heat = fig.add_heatmap(z=np.zeros((Z,Z)), colorscale="Viridis", zmax=10, zmin=0)

# make the figure a square
display(fig)
minvs = []
maxvs = []

# frame interval slider (interactive)
frame_interval = w.IntSlider(value=100, min=1, max=100, step=1, description="Frame N", continuous_update=True)

# async display thread
viz_q = queue.Queue(maxsize=2)
stop_flag = threading.Event()

# double-buffer latest frame (lock-protected)
latest_frame = {"data": None, "t": -float("inf")}
latest_lock = threading.Lock()


def viz_loop():
    last_drawn_t = -float("inf")
    while not stop_flag.is_set():
        try:
            # Always consume to the newest frame to minimize latency
            item = viz_q.get(timeout=0.02)
            while True:
                try:
                    item = viz_q.get_nowait()
                except queue.Empty:
                    break
            with latest_lock:
                latest_frame["data"], latest_frame["t"] = item
        except queue.Empty:
            pass

        with latest_lock:
            mP_grid = latest_frame["data"]
            cur_t = latest_frame["t"]

        if mP_grid is not None and cur_t >= last_drawn_t:
            with fig.batch_update():
                fig.data[0].z = mP_grid
                # Avoid relayout cost each frame; title only when significantly changed
            last_drawn_t = cur_t

        # tiny sleep to yield to UI thread
        time.sleep(0.01)


viz_thread = threading.Thread(target=viz_loop, daemon=True, name="viz_thread")
viz_thread.start()
display(frame_interval)

# simulation loop
try:
    for tick in range(100_000):
        update_I(I, V, weights, t)
        t, dR = step_simulation(R, V, I, t, dt=1e-3)
        update_W(weights, R, dR)

        N = max(1, int(frame_interval.value))
        if tick % N == 0:
            mP_grid = R.reshape(Z,Z)
            # Avoid extra copy; let viz thread overwrite on next frame
            try:
                viz_q.put_nowait((mP_grid, t))
            except queue.Full:
                # Drop oldest then enqueue latest to reduce latency
                try:
                    _ = viz_q.get_nowait()
                except queue.Empty:
                    pass
                finally:
                    try:
                        viz_q.put_nowait((mP_grid, t))
                    except queue.Full:
                        pass
        time.sleep(0.001)
finally:
    stop_flag.set()
    viz_thread.join(timeout=1)


RuntimeWarning: overflow encountered in scalar multiply

In [ ]:
i=7
j=np.int64(list(weights.children[i]))
weights[[i],j]

In [ ]:
V

In [ ]:
weights[np.arange(10),np.arange(10,20)]

In [93]:
# set the IOPub message limit much higher
import os, json, sys

# Increase limits for the running IPython kernel process (best effort)
os.environ["IPYKERNEL_CELL_NAME"] = "high_iopub_limits"
try:
    from IPython import get_ipython

    ip = get_ipython()
    if ip is not None and hasattr(ip, "kernel") and hasattr(ip.kernel, "session"):
        # These config keys are read at startup; for a running kernel we adjust traitlets directly if present
        if hasattr(ip.kernel, "iopub_thread") and hasattr(ip.kernel.iopub_thread, "rate_limit"):
            # Disable rate limiting by setting huge limits
            ip.kernel.iopub_thread.rate_limit = 1e10
            ip.kernel.iopub_thread.max_msg_rate = 1e9
            ip.kernel.iopub_thread.max_msg_size = int(1e9)
except Exception as e:
    print("Could not adjust IOPub limits at runtime:", e, file=sys.stderr)

# Also tell Jupyter Server (if it respects env for spawned kernels later in this session)
os.environ["IPKernelApp.iopub_msg_rate_limit"] = "1000000000"
os.environ["IPKernelApp.iopub_data_rate_limit"] = "1.0e11"


In [90]:
ip.kernel.iopub_thread.rate_limit = 1e10

In [89]:
os.environ["IPKernelApp.iopub_msg_rate_limit"] = "1000000000"

In [ ]:
import numpy as np, time
import plotly.graph_objects as go

fig = go.FigureWidget([go.Scatter(x=[], y=[], mode="lines")])
display(fig)

y = []
for t in range(1000):
    y.append(np.sin(t/10))
    with fig.batch_update():
        fig.data[0].x = np.arange(len(y))
        fig.data[0].y = y
    time.sleep(0.001)


In [ ]:
 import ipywidgets as w, plotly.io as pio
print("ipywidgets", w.__version__)
_ = w.IntSlider()  # should render a slider
